In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# A network instead of a table: deep Q-learning

## Where this session fits

> **The limitation we start from: Q-learning keeps one number per state and action**

- on real problems there are far too many states for a table

- **The question:** what replaces the table, and what new problems does the replacement bring?
- **This session's goal:** scale Q-learning to large state spaces with a network, and see where that path stops
- **Position: Part A (value-based methods), session 5 (the last method of this part)**
    - values and the Bellman equation, dynamic programming, learning from samples, acting with Q-learning and SARSA, then **a network instead of a table (this session)**
    - learning the policy directly is a separate part that follows
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. say what a network adds over a table (it *generalizes*) and what it costs
2. write the deep Q-learning target and loss, and compute them for a mini-batch
3. explain why *replay* is allowed for Q-learning and not for SARSA
4. explain why the target network is lagged
5. compute the loss of a mini-batch by hand, and say which number is the prediction and which is the target


**Recall: the Q-learning update of the last session**

> **Q-learning moves one table entry toward a target built from one sample.**

$$
\actvalfun(\state, \act) \leftarrow \actvalfun(\state, \act) + \alpha \bigl[ \rew + \disc \max{\act'} \actvalfun(\state', \act') - \actvalfun(\state, \act) \bigr]
$$
- the table entry $\actvalfun(\state, \act)$ is the *prediction*
- the bracketed target is $\rew + \disc \max{\act'} \actvalfun(\state', \act')$ (zero max term if $\state'$ is terminal)
- this session keeps the update and replaces the table by a network

# Why a table stops working

**Why this section?** Last session learned $\actvalfun$ as a table. The table was the limitation. To replace it we first need to see exactly what fails.

> **A table fails in two ways: size and no sharing. A function with parameters fixes both.**

- Frozen Lake: $16$ states $\times$ $4$ actions $=$ $64$ numbers: a table is fine
- Microduck, the walking robot of the tour: the state is $61$ *continuous* numbers (joint angles and speeds)
- **Failure 1: size.** With continuous numbers the states cannot even be counted, let alone given one entry each
- **Failure 2: no sharing.** A table learns each entry alone
    - a state never visited has no value at all
    - two nearly identical states learn nothing from each other

**A function with parameters fixes both**

- it has a fixed size, however many states there are
- it **generalizes**: similar states get similar values, so what is learned at one state helps at its neighbors
- what the function outputs, and for which problems, is the topic of the next slide

**What the output is: a value per action, or the action itself**

<img src="images/a5_outputs.png" width=80%>

- **Q-network** (this session): one output per action of a fixed, finite list
    - the same list for every state
    - each output is the value of that action in the input state
- **policy network** (Part B): the output is the action itself
    - Microduck's action is a vector of $14$ continuous joint commands
    - one action, not $14$ values and not $14$ actions

**Why Microduck has no Q-network**

- a Q-network needs a finite list of actions, one output each
- a continuous $14$-vector action has infinitely many actions: there is no list
- so we learn the policy directly: the last session of Part A and Part B


## Three tables a network could replace

**Why this section?** A network can replace a table, but which one? The Q-table of the last session is one of three tables in this series, and the choice shapes what follows.

> **A network can replace any of three tables: action values (this session), state values, or the policy itself.**

<table>
<thead><tr><th nowrap><div align="left">The table being replaced</div></th><th nowrap><div align="left">Replaced by</div></th><th nowrap><div align="left">What it gives</div></th><th nowrap><div align="left">Where</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">action values $\actvalfun(\state, \act)$</div></td><td nowrap><div align="left">$\actvalfun_\theta(\state)$: takes only $\state$, outputs the value of every action</div></td><td nowrap><div align="left">the policy by argmax (<i>value-based</i>)</div></td><td nowrap><div align="left"><b>this session</b> (deep Q-learning)</div></td></tr>
<tr><td nowrap><div align="left">state values $\statevalfun(\state)$</div></td><td nowrap><div align="left">$\statevalfun_\theta(\state)$</div></td><td nowrap><div align="left">a <i>judge</i> of the policy, not a policy</div></td><td nowrap><div align="left">Part B (a "critic", a learned judge of states)</div></td></tr>
<tr><td nowrap><div align="left">the policy itself $\pi(\act \mid \state)$</div></td><td nowrap><div align="left">$\pi_\theta(\act \mid \state)$</div></td><td nowrap><div align="left">the policy, with no argmax</div></td><td nowrap><div align="left">Part B</div></td></tr>
</tbody></table>

**What differs between the three**

- the network is the same idea each time: **function approximation**, with parameters $\theta$ trained by gradient steps
- what differs is *which quantity* the network represents, and so what it is trained to match
- this session takes the first row: it is the smallest change to what we already have, since Q-learning already learns $\actvalfun$
- the last session of Part A (the methods table) asks what the first row cannot do, and the answer is the third row
- none of the three is a *learned model* of the environment, which is a different idea we do not cover here


## The Q-network: input, output, and what it is compared with

**Why this section?** A network is defined by what goes in, what comes out, and how its output is compared with a label. Here they are for the Q-network.

> **The input is the state. The output is one predicted value per action. The loss compares the output of the action taken with a label, the Bellman target $y$.**

- the network's output takes the place of the table entry: there is no table to match
- it is moved toward the same target $y$ that the table entry would have been moved toward, were the table to exist
    - one difference: the target's next-state value $\max{\act'}$ now comes from a network, not from table entries


**Q-table versus Q-network**

<table>
<thead><tr><th nowrap><div align="left"></div></th><th nowrap><div align="left">Q-table</div></th><th nowrap><div align="left">Q-network</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">input</div></td><td nowrap><div align="left">the state $\state$ selects a row</div></td><td nowrap><div align="left">the state $\state$ only, never the action</div></td></tr>
<tr><td nowrap><div align="left">output</div></td><td nowrap><div align="left">the stored row: $\actvalfun(\state, \act)$ for every action $\act$</div></td><td nowrap><div align="left">the list $\actvalfun_\theta(\state)$: one predicted value for each action of a fixed list, computed by the network from $\state$ alone; all of it is used, for acting and for the target's max</div></td></tr>
<tr><td nowrap><div align="left">label</div></td><td nowrap><div align="left">the target $y$ for the entry just visited, $(\state, \act)$: one number built from $\rew$ and $\state'$</div></td><td nowrap><div align="left">the same target $y$, for the output of the action taken: one number, not a list; the other outputs get no label</div></td></tr>
<tr><td nowrap><div align="left">learning step</div></td><td nowrap><div align="left">move the stored entry <i>part of the way</i> toward $y$ (step size $\alpha$)</div></td><td nowrap><div align="left">move the network's output for the action taken <i>part of the way</i> toward $y$ (one gradient step, step size the learning rate)</div></td></tr>
<tr><td nowrap><div align="left">loss</div></td><td nowrap><div align="left">none: a direct update with step size $\alpha$</div></td><td nowrap><div align="left">$\bigl(y - \actvalfun_\theta(\state, \act)\bigr)^2$, then a gradient step on $\theta$</div></td></tr>
</tbody></table>


**Why the output is a list when the loss uses one entry**

> **One transition trains one output, but the whole list is needed to act and to build the target.**

- **acting:** to choose the best action in $\state$ we need the value of every action: one forward pass gives them all
- **the target:** $y$ contains $\max{\act'} \actvalfun_{\theta^-}(\state', \act')$, a maximum over every output at the next state
- **training:** a transition trains the output of its own action
    - different transitions train different outputs
    - the shared layers learn from all of them
- the alternative, a network that takes $(\state, \act)$ and outputs one number, needs one forward pass per action for each of those maxima


**Two things to pin down: label and notation**

- **label** is the supervised-learning word for the number the prediction is meant to match
    - here the label is the target $y$, a scalar for the one action taken
        - it is built from $\rew$ and $\state'$, not given as data
    - it is an *estimate* of the true value, not the true value
        - it can be biased and noisy
        - neither the table entry nor the network's output is expected to reach it in one step
- notation: $\actvalfun_\theta(\state, \act)$ means the entry of the list $\actvalfun_\theta(\state)$ at the position of action $\act$
    - the action *selects an output to read*: it is not an input of the network

**The Q-network in a picture**

<img src="images/a5_qnetwork.png" width=68%>

- the network has one output per action, so in this simple form the actions must be *discrete* (a short list)
- the *prediction* $\hat y$ is the output for the action actually taken
- the *target* $y$ is the label: the reward plus the discounted best value at the next state
- only the highlighted output is in the loss, so the other outputs are not pushed

**The loss for one transition**

> **For one step $(\state, \act, \rew, \state')$ the loss compares the prediction $\hat y$ with the target $y$.**

$$
L(\theta) = \Bigl( \underbrace{\rew + \disc \max{\act'} \actvalfun_{\theta^-}(\state', \act')}_{\text{target } y} \;-\; \underbrace{\actvalfun_\theta(\state, \act)}_{\text{prediction } \hat y} \Bigr)^2
$$

- $\theta$ are the parameters being trained, often called the network's *weights*: the prediction $\hat y$ is the network's output at the action taken
- **naming note:** this deck says *weights* for $\theta$ in some places and *parameters* in others
    - both mean $\theta$: *weights* is the usual word for a network, *parameters* is the general word
    - Part B says only *parameters*: there *weight* names another number, the multiplier on a gradient term
- $\theta^-$ is a lagged copy of $\theta$, used only in the target (explained under Problem 2 below)
- at a terminal $\state'$ the target is just $y = \rew$
- the target is Q-learning's target, unchanged
- **one gradient step** on $\theta$ lowers the loss, moving $\hat y$ toward $y$

**What kind of number the target is**

- $y$ is treated as a constant in each gradient step
    - the gradient flows through $\hat y$ (the weights $\theta$)
    - not through $y$ (the weights $\theta^-$)
- $y$ is a sample of the **one-step form** (a bootstrapped sample), as in TD(0)
    - biased while the network is wrong
    - lower variance than a Monte Carlo return


# Problem 1: consecutive data are correlated

**Why this section?** Training a network by gradient steps assumes examples that look like independent draws. An agent's experience does not.

> **Consecutive transitions of one episode are strongly correlated, and the data drift as the policy changes. The patch is replay memory, on the next slide.**

- consecutive transitions of one episode are strongly correlated: similar states, one after another
- the data also *drift*: as the policy changes, the states it visits change
- a network trained on the latest few steps swings toward them and forgets older experience
    - *catastrophic forgetting*: learning the new overwrites what was learned about the old


## Replay memory: store transitions, draw them at random

**Why this section?** The fix for correlated data: break the order of the episode.

> **Store every transition as it happens, then train on mini-batches drawn uniformly at random from the store, not in the order of the episode.**

- a **transition** is one step $(\state, \act, \rew, \state')$ plus a flag for "terminal"
- the **replay memory** stores them: a fixed-size first-in-first-out store, where the oldest transitions are dropped when it is full
    - typical capacity $N$: $10^5$ to $10^6$ transitions, spanning many episodes
    - it also holds data from older versions of the policy, since the policy changes while the memory fills
- a **mini-batch** is a small set of transitions drawn *uniformly at random* from the memory
    - one batch mixes steps from many episodes and many older policies
- the loss is the mean over the mini-batch of the one-transition loss, and one gradient step lowers it
- why a batch and not one transition per update?
    - as for any network, the gradient averaged over several examples is a smoother (lower-variance) estimate than the gradient of one example
    - and computing it is more efficient
    - here the random draw also decorrelates the examples in each batch
- a second reason: the weights are shared, so a step for one transition changes the predictions at many other states
    - in a table, an update changes one entry only
    - a diverse batch makes the step a compromise across many states, not a fit to the current one

**Replay memory in the loop**

> **In: one new transition per step, at the newest end. Out: a mini-batch drawn uniformly at random.**

<img src="images/a5_replay_mechanism.png" width=70%>

**What replay gives us**

- **less correlated:** random draws break the order of an episode
    - not truly independent: neighbors from the same episode are still in the store
    - but far closer to the independent examples that gradient steps assume
- **reused:** each transition is used many times
    - better *sample efficiency*: more learning per step in the environment
- **older policies:** a batch mixes transitions from several earlier policies, not only the current one
    - this raises a question (two slides on)

<img src="images/a5_replay_batch.png" width=85%>


A mini-batch by numbers ($\disc = 0.9$, two transitions)

<table>
<thead><tr><th nowrap><div align="left"></div></th><th nowrap><div align="center">reward</div></th><th nowrap><div align="left">next state</div></th><th nowrap><div align="center">$\max{\act'} \actvalfun_{\theta^-}$ at the next state</div></th><th nowrap><div align="center">target $y$</div></th><th nowrap><div align="center">prediction $\hat y = \actvalfun_\theta(\state, \act)$</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">1</div></td><td nowrap><div align="center">$+1$</div></td><td nowrap><div align="left">terminal</div></td><td nowrap><div align="center">(none)</div></td><td nowrap><div align="center">$1$</div></td><td nowrap><div align="center">$0.6$</div></td></tr>
<tr><td nowrap><div align="left">2</div></td><td nowrap><div align="center">$0$</div></td><td nowrap><div align="left">not terminal</div></td><td nowrap><div align="center">$0.8$</div></td><td nowrap><div align="center">$0 + 0.9 \times 0.8 = 0.72$</div></td><td nowrap><div align="center">$0.5$</div></td></tr>
</tbody></table>

- loss $=$ the mean over the batch of $2$ $= \frac{1}{2}\left[(1 - 0.6)^2 + (0.72 - 0.5)^2\right] \approx 0.10$
- one gradient step on $\theta$ lowers this loss, moving the predictions $0.6$ and $0.5$ toward $1$ and $0.72$
- $y$ is held fixed
- the predictions are at the stored states and actions
- a mini-batch is used for one gradient step


## Why replay is allowed: the payoff of off-policy

**Why this section?** A stored transition was generated by an *older* policy. Last session asked which policy an update learns about. Here is why that matters.

> **Replay is legal for Q-learning (off-policy) and not for SARSA (on-policy): off-policy learning lets us reuse data.**

- **on-policy:** the update learns about the policy that generated the data
- **off-policy:** the data came from a different policy than the one being learned about (defined last session)
- a transition in the memory came from the behavior policy in force when it was stored: not the current one
- so replay is legitimate only if the update does not care which policy produced the data

**Why Q-learning can reuse old data**

- **Q-learning** (off-policy)
    - its one-step target $\rewseq + \disc \max{\act'} \actvalfun(\state', \act')$ uses the *best* next action, not the action the old policy took
    - the reward and next state depend only on the environment given $(\state, \act)$, not on who chose $\act$
    - so each stored transition is a valid sample of the optimality backup, and replay is legal
    - for a table, reusing old transitions this way is exact
    - with a network the same reuse is standard practice, still without a convergence guarantee
    - one condition: the old policies must have *tried* the relevant state-action pairs, since the update only improves $\actvalfun$ where the memory has data

**SARSA cannot reuse old data**

> **SARSA's target uses the next action chosen by the policy being learned about. An old transition holds an action chosen by an older policy, so the target would be about the wrong policy.**

- **SARSA** (on-policy)
    - its target uses the next action $\act'$ chosen by the policy being learned about
    - an old transition holds an $\act'$ chosen by an *older* policy: the target is then about the wrong policy
    - so SARSA cannot reuse old data without a correction
        - importance weights: not covered here, rarely used

**The two targets, side by side**

<img src="images/a5_qlearning_vs_sarsa_target.png" width=80%>

- so the definition from last session has a purpose: **off-policy learning lets us reuse data**, and reuse is what makes network training practical
- the same payoff returns in Part B, where old data are reused through a *ratio* of probabilities


# Problem 2: the target moves, and the lagged network fixes it

**Why this section?** The target $y$ is computed with a network, and that network is the one being trained. That makes the target move.

> **The loss compares two numbers that both come from a network: the prediction $\hat y$ and the target $y$ (the label). When $\theta$ changes, both change.**

- **prediction** $\hat y = \actvalfun_\theta(\state, \act)$: the output of the network being trained, at the stored state and action
- **target** $y = \rew + \disc \max{\act'} \actvalfun_{\theta^-}(\state', \act')$: the label, built from a network's output at the next state
- in supervised learning the label is fixed data, here it is built from the network's own predictions
- $y$ is written with $\theta^-$, a lagged copy of $\theta$ introduced below
- first suppose $\theta^-$ is simply $\theta$

**Why the target moves**

> **In supervised learning the label is fixed, so a step toward it can fit that example. Here the label is built from $\theta$, so the step moves the label too.**

- **supervised learning:** the label $y$ is fixed data
    - a gradient step pulls the prediction $\hat y$ toward $y$
    - a large enough step would make this example's prediction perfect
- **here:** with $\theta^-$ replaced by the current weights $\theta$, $y = \rew + \disc \max{\act'} \actvalfun_\theta(\state', \act')$
    - the step that moves $\hat y = \actvalfun_\theta(\state, \act)$ also changes $\actvalfun_\theta(\state', \cdot)$, because the weights are shared
    - so $y$ shifts as well, and the example is *not* fit after the step
- the step treats $y$ as a constant, but $y$ does not stay constant

**The chase, and why it can fail**

- the network chases its own output: a moving prediction and a moving target at once
- this can oscillate or diverge
    - bootstrapped targets, a network, and off-policy data together are known to be unstable

**The fix: a target network**

> **There are two sets of weights for one network architecture: $\theta$ makes the predictions and is trained, and $\theta^-$, a lagged copy, only computes the targets.**

- the **main network** has weights $\theta$
    - it makes the predictions $\hat y$
    - it is updated by the gradient steps
    - the agent acts with it
- the **target network** has weights $\theta^-$, a lagged copy of $\theta$
    - it computes $y$ only, and no gradient step changes it
    - it is refreshed from the main network every $C$ steps, so between refreshes the target holds still
- the lag *reduces* the chase, it does not remove the instability


**The lagged target holds still: a picture**

<img src="images/a5_target_chase.png" width=85%>

- both fixes are patches around one fact: we train a network on bootstrapped targets, from data the agent itself produces
    - a table update changes only one entry
    - a network update changes the output at every state at once


## The deep Q-learning loop

**Why this section?** All the pieces fit in one loop: act, store, sample, learn, sometimes sync.

<img src="images/dqn_loop.png" width=90%>

- the main network acts ($\epsilon$-greedy) and is trained
- the target network only provides $y$
- every $C$ steps the target network is refreshed with the main weights


**Pseudo-code for deep Q-learning** (the loop of Mnih et al., 2015; it is not in the book)

- initialize
    - a replay memory $D$ of capacity $N$
    - a main network $\actvalfun_\theta$ with random weights $\theta$
    - a target network $\actvalfun_{\theta^-}$ with weights $\theta^- = \theta$
- set the exploration rate $\epsilon$ (annealed from $1$ to a small value), the discount $\disc$, and the learning rate $\alpha$ (for the optimizer)
- for each episode
    - initialize the state $\state$
    - for each step, until $\state$ is terminal
        - **(1) act:** with probability $\epsilon$ choose a random action $\act$
            - otherwise $\act = \argmax{\act'} \actvalfun_\theta(\state, \act')$
            - take $\act$, observe $\rew$ and $\state'$
        - **(2) store:** put the transition $(\state, \act, \rew, \state', \text{done})$ in $D$
            - *done* says whether $\state'$ is terminal
            - it is one step, not an episode
            - when $D$ is full, the oldest transition is dropped
        - **(3) learn:** once $D$ holds enough transitions, sample a random mini-batch of transitions $(\state_j, \act_j, \rew_j, \state'_j, \text{done}_j)$ from $D$
            - target: $y_j = \rew_j$ if $\text{done}_j$ (that is, $\state'_j$ is terminal), otherwise $y_j = \rew_j + \disc \max{\act'} \actvalfun_{\theta^-}(\state'_j, \act')$ ($y_j$ is a constant: no gradient through it)
            - loss: the mean over the batch of $\bigl(y_j - \actvalfun_\theta(\state_j, \act_j)\bigr)^2$; take a gradient step on $\theta$
        - **(4) sync:** every $C$ environment steps, $\theta^- \leftarrow \theta$
        - $\state \leftarrow \state'$ (after the four parts)


## Aside: numbers from the original deep Q-learning paper, and a code example (for your own review)

- replay memory $N = 10^6$ transitions, target network refreshed every $C = 10^4$ steps, $\epsilon$ annealed from $1$ to $0.1$
- (source: Mnih et al., 2015, *Nature*) one convolutional network reached human-level scores on many of 49 Atari games from the screen pixels alone
- these hyperparameters mattered: deep Q-learning is sensitive to them, and failed runs are common
- [Here](RL_OnPolicy_vs_OffPolicy_code_examples.ipynb) is a notebook with Q-learning with and without replay memory, and SARSA


## Where deep Q-learning sits on the threads

**Why this section?** Each method is placed on the same few questions (the *threads*: model, what is learned, on- or off-policy, backup, bias, variance). Here is this method. The full methods table follows this session.

| Thread | Deep Q-learning |
|:--|:--|
| model needed? | no: it learns from sampled transitions |
| what is learned | action values; the policy is read off by argmax |
| on- or off-policy | **off**: replay reuses old data |
| backup | one sampled next state, one step deep (like TD(0)) |
| bias of the target | biased while the network is wrong (bootstrapped) |
| variance of the target | lower than Monte Carlo (one step) |
| data correlation | consecutive transitions are correlated; replay breaks that |
| the table | replaced by a network: it generalizes |


# What we established, and what it sets up

- **Established**
    - a table fails on size and on sharing
    - a network of parameters $\theta$ generalizes across states
    - a network can replace three tables: action values (this session), state values, or the policy (Part B)
    - deep Q-learning trains $\actvalfun_\theta$ by regression on Q-learning's target
        - the aim is still $\actvalfun_*$ while acting $\epsilon$-greedily
        - with a network, convergence is no longer guaranteed
    - **replay** reuses old transitions, and is legal *because* Q-learning is off-policy
        - SARSA cannot without a correction
    - the **lagged target network** reduces the bootstrapped target's chase of the network
    - a network of action values needs a short list of discrete actions: the last session of Part A says where that stops
- **Sets up**
    - Part B: policy gradient. The network goes on the policy, and the objective is the expected return itself
    - bias and variance, introduced for the targets of Monte Carlo and TD, return there as the central difficulty
    - off-policy reuse returns as the ratio of probabilities in PPO (a policy-gradient method)
    - the table of methods against threads (model needed, on- or off-policy, backup, bias, variance) closes the value-based path
